In [1]:
import numpy as np
from numpy.linalg import LinAlgError
import scipy
from datetime import datetime
from collections import defaultdict
from scipy.special import expit
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import line_search, minimize_scalar
from numpy.linalg import norm
import time
from scipy.sparse import diags
import seaborn as sns
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import StandardScaler
from pathlib import Path

## 1. Реализация основных методов и функций для последующего анализа

In [2]:
class BaseSmoothOracle(object):
    """
    Base class for implementation of oracles.
    """
    def func(self, x):
        """
        Computes the value of function at point x.
        """
        raise NotImplementedError('Func oracle is not implemented.')

    def grad(self, x):
        """
        Computes the gradient at point x.
        """
        raise NotImplementedError('Grad oracle is not implemented.')

    def hess(self, x):
        """
        Computes the Hessian matrix at point x.
        """
        raise NotImplementedError('Hessian oracle is not implemented.')

In [3]:
class LogRegL2Oracle(BaseSmoothOracle):
    """
    Oracle for logistic regression with l2 regularization:
         func(x) = 1/m sum_i log(1 + exp(-b_i * a_i^T x)) + regcoef / 2 ||x||_2^2.

    Let A and b be parameters of the logistic regression (feature matrix
    and labels vector respectively).
    """
    def __init__(self, A, b, regcoef):
        self.matvec_Ax = lambda x: A@x
        self.matvec_ATx = lambda x: A.T@x
        self.matmat_ATsA = lambda x: A.T @ x @ A
        self.b = b
        self.regcoef = regcoef

    def func(self, x):
        m = self.b.shape[0]
        return np.dot(np.ones(m), np.logaddexp(0, -self.b * (self.matvec_Ax(x))))/m + self.regcoef*np.dot(x, x)/2

    def grad(self, x):
        m = self.b.shape[0]
        return -self.matvec_ATx(self.b * scipy.special.expit(-self.b*self.matvec_Ax(x)))/m + self.regcoef*x

    def hess(self, x):
        m = self.b.shape[0]
        return self.matmat_ATsA(diags(scipy.special.expit(-self.b*self.matvec_Ax(x))*scipy.special.expit(self.b*self.matvec_Ax(x))))/m + self.regcoef*np.eye(x.shape[0])

In [4]:
class search_alpha(object):
  """
  Step finder satysfying the Armijo-Wolfe conditions with c1, c2 coefficients.
  """
  def __init__(self, c1=0.1, c2=0.9):
    self.c1 = c1
    self.c2 = c2

  def backtracking(self, oracle, x_k, d_k):
    alpha_0 = 0
    phi_zero = oracle.func(x_k)
    phi_grad = oracle.grad(x_k).dot(d_k)
    while (oracle.func(x_k+d_k*alpha_0) > phi_zero+alpha_0*self.c1*phi_grad):
        alpha_0 /= 2
    return alpha_0

  def search(self, oracle, x_k, d_k):
    next_alpha = line_search(oracle.func, oracle.grad, x_k, d_k, c1=self.c1, c2=self.c2)[0]
    if (next_alpha==None):
        return self.backtracking(oracle, x_k, d_k)
    else:
        return next_alpha

In [5]:
def BFGS(oracle, x_0, iter, tol, c1, c2):
  """
  Implementation of BFGS method, finds saddle point and value of oracle.func in it
  """
  x = np.array(x_0, dtype=float)
  n = len(x)
  H = np.eye(n)

  f = oracle.func(x)
  grad = oracle.grad(x)
  grad_norm_x0 = np.dot(grad, grad)
  est = grad_norm_x0 * tol

  search_tool = search_alpha(c1, c2)

  for k in range(iter):
    if np.dot(grad, grad) < est:
      return x, f

    d = - H @ grad
    if search_tool.search(oracle, x, d) is not None:
      alpha = search_tool.search(oracle, x, d)
    else:
      alpha = 1.0

    x_new = x + alpha*d
    f_new = oracle.func(x_new)
    grad_new = oracle.grad(x_new)

    s = x_new - x
    y = grad_new - grad

    ys_dot = np.dot(y,s).astype(float)
    rho = 1.0 / ys_dot
    Hy = H @ y

    term1 = ((ys_dot + np.dot(y, Hy).astype(float)) / (ys_dot ** 2)) * np.outer(s, s)
    term2 = (np.outer(Hy, s) + np.outer(s, Hy)) / ys_dot

    H += term1 - term2
    H += H.T
    H /= 2

    x, f, grad = x_new, f_new, grad_new
  return x, f


In [6]:
def BFGS_greed(oracle, x_0, iter, tol, c1, c2):
  """
  Greedy quasi-Newton update.
  """
  x = np.array(x_0, dtype=float)
  n = len(x)
  L = np.eye(n)

  f = oracle.func(x)
  grad = oracle.grad(x)
  grad_norm_x0 = np.dot(grad, grad)
  est = grad_norm_x0 * tol

  search_tool = search_alpha(c1, c2)

  for k in range(iter):
    if np.dot(grad, grad) < est:
      return x, f

    d = - (L.T @ (L @ grad))

    if search_tool.search(oracle, x, d) is not None:
      alpha = search_tool.search(oracle, x, d)
    else:
      alpha = 1.0

    x_new = x + alpha * d
    f_new = oracle.func(x_new)
    grad_new = oracle.grad(x_new)

    s = x_new - x
    y = grad_new - grad

    A = oracle.hess(x_new)
    best_val = -np.inf
    best_i = 0

    for i in range(n):
      e = np.zeros(n)
      e[i] = 1.0
      u = L.T @ e
      Au = A @ u
      val = u @ Au
      if (val > best_val):
        best_val = val
        best_i = i
    tilde_u = np.zeros(n)
    tilde_u[best_i] = 1.0
    u = L.T @ tilde_u

    Au = A @ u
    denom = u @ Au
    v = np.sqrt(denom) * tilde_u
    L -= np.outer(L @ Au - v, u) / denom

    x, f, grad = x_new, f_new, grad_new
  return x, f


def BFGS_rnd(oracle, x_0, iter, tol, c1, c2, seed=None):
  """
  Randomized quasi-Newton update.
  """
  x = np.array(x_0, dtype=float)
  n = len(x)
  L = np.eye(n)

  f = oracle.func(x)
  grad = oracle.grad(x)
  grad_norm_x0 = np.dot(grad, grad)
  est = grad_norm_x0 * tol

  search_tool = search_alpha(c1, c2)

  rng = np.random.default_rng(seed)

  for k in range(iter):
    if np.dot(grad, grad) < est:
      return x, f

    d = - (L.T @ (L @ grad))

    alpha = search_tool.search(oracle, x, d)
    if search_tool.search(oracle, x, d) is not None:
      alpha = search_tool.search(oracle, x, d)
    else:
      alpha = 1.0

    x_new = x + alpha * d
    f_new = oracle.func(x_new)
    grad_new = oracle.grad(x_new)

    s = x_new - x
    y = grad_new - grad

    A = oracle.hess(x_new)
    tilde_u = rng.standard_normal(n)
    tilde_norm = np.linalg.norm(tilde_u)
    if tilde_norm < 1e-16:
      tilde_u[0] = 1.0
      tilde_norm = 1.0
    u = L.T @ tilde_u

    # rho = 1.0 / (y @ s)

    Au = A @ u
    denom = np.dot(u, Au)
    v_k = np.sqrt(max(0.0, denom)) * (tilde_u / tilde_norm)
    num = (L @ Au) - v_k
    L -= np.outer(num, u) / denom

    x, f, grad = x_new, f_new, grad_new
  return x, f


In [7]:
!wget https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/w8a

--2026-02-02 04:39:14--  https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/w8a
Resolving www.csie.ntu.edu.tw (www.csie.ntu.edu.tw)... 140.112.30.26
Connecting to www.csie.ntu.edu.tw (www.csie.ntu.edu.tw)|140.112.30.26|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3478238 (3.3M)
Saving to: ‘w8a.4’

w8a.4               100%[===================>]   3.32M  4.29MB/s    in 0.8s    

2026-02-02 04:39:16 (4.29 MB/s) - ‘w8a.4’ saved [3478238/3478238]



In [8]:
data_path = Path("w8a")
X, y = load_svmlight_file(data_path)
X = X.toarray()
y = y.astype(float)

y[y == 0] = -1
scaler = StandardScaler()
X = scaler.fit_transform(X)

n_samples, n_features = X.shape
print(f"Loaded w8a: n={n_samples}, d={n_features}")

regcoef = 1e-4
oracle = LogRegL2Oracle(X, y, regcoef)

x0 = np.zeros(n_features)
max_iter = 100
c1, c2 = 1e-4, 0.9
tol = 1e-6

results = {}


def run(method_name, method_fn, **kwargs):
  print(f"\nRunning {method_name}...")
  start = time.time()
  x_star, f_star = method_fn(oracle, x0, max_iter, tol, c1, c2, **kwargs)
  elapsed = time.time() - start
  grad_norm = np.linalg.norm(oracle.grad(x_star))
  results[method_name] = {
  "f": f_star,
  "grad_norm": grad_norm,
  "time": elapsed,
  }
  print(f" f(x*) = {f_star:.6e}")
  print(f" ||grad|| = {grad_norm:.3e}")
  print(f" time = {elapsed:.2f} sec")

run("BFGS", BFGS)
run("Greedy-QN", BFGS_greed)
run("Random-QN", BFGS_rnd, seed=42)


print("\n===== Summary =====")
for k, v in results.items():
  print(f"{k:12s} | f = {v['f']:.6e} | ||grad|| = {v['grad_norm']:.3e} | time = {v['time']:.2f}s")

Loaded w8a: n=49749, d=300

Running BFGS...
 f(x*) = 1.483475e-01
 ||grad|| = 1.317e-02
 time = 43.32 sec

Running Greedy-QN...
 f(x*) = 4.458812e-01
 ||grad|| = 6.912e-02
 time = 110.01 sec

Running Random-QN...
 f(x*) = 2.718911e-01
 ||grad|| = 2.404e-02
 time = 123.50 sec

===== Summary =====
BFGS         | f = 1.483475e-01 | ||grad|| = 1.317e-02 | time = 43.32s
Greedy-QN    | f = 4.458812e-01 | ||grad|| = 6.912e-02 | time = 110.01s
Random-QN    | f = 2.718911e-01 | ||grad|| = 2.404e-02 | time = 123.50s
